# Parte 2 — Modelos de aprendizaje supervisado

**Caso B — Predicción de precios de viviendas (Ames Iowa Housing).** Regresión sobre `SalePrice`.

Reglas de trabajo, heredadas del README del proyecto (secciones 4 y 6.3):

- Se usa el mismo split de la Parte 1 (80 % entrenamiento, 20 % prueba, `random_state=42`).
- La selección de modelos e hiperparámetros se hace **solo con validación cruzada sobre entrenamiento**.
- El conjunto de prueba se usa **una única vez**, en la evaluación final (sección 7).
- Las metas de negocio se fijaron antes de comparar modelos y no se modifican a partir de los resultados.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

directorio_actual = Path.cwd()

if (directorio_actual / "src").is_dir():
    ROOT = directorio_actual
elif (directorio_actual.parent / "src").is_dir():
    ROOT = directorio_actual.parent
else:
    raise FileNotFoundError(
        "Abre el notebook desde la raíz del proyecto "
        "o desde su carpeta notebooks."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import load_data, load_raw
from src.train import (
    RANDOM_STATE, TOLERANCIA, META_COBERTURA, META_MEJORA_MAE,
    COLUMNAS_EXCLUIDAS, cargar_datos_crudos, calcular_metricas,
)

IMAGES_DIR = ROOT / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Datos de entrada

El modelado usa los **datos crudos ya limpios** (`cargar_datos_crudos()`), no los CSV transformados de
`data/processed/`. La razón es metodológica: en validación cruzada el imputador, el escalador y el
codificador deben ajustarse dentro de cada partición de entrenamiento (README, secciones 6.3 y 10.4).
Los CSV transformados sirven aquí solo para comprobar que el split coincide con el de la Parte 1.

In [ ]:
X_train, X_test, y_train, y_test = cargar_datos_crudos()

# Comprobación contra la interfaz compartida de la Parte 1 (mismo split y mismo orden)
Xp_train, Xp_test, yp_train, yp_test = load_data()
assert len(X_train) == len(Xp_train) and len(X_test) == len(Xp_test)
assert np.allclose(y_train.to_numpy(), yp_train.to_numpy())
assert np.allclose(y_test.to_numpy(), yp_test.to_numpy())

pd.Series({
    "registros_entrenamiento": len(X_train),
    "registros_prueba": len(X_test),
    "predictores_antes_de_transformar": X_train.shape[1],
    "predictores_transformados_Parte_1": Xp_train.shape[1],
})

## 2. Variables excluidas de los predictores

Se eliminan `Sale Type` y `Sale Condition` antes de entrenar. Describen las condiciones de la transacción
(tipo de contrato, venta parcial, ejecución hipotecaria, etc.) y solo se conocen al cerrar la venta. Una
estimación de valoración debe usar información disponible **antes** de la venta (README, sección 5).

- Ambas variables se usan después, únicamente como **diagnóstico** del análisis de errores (sección 7).
- Se conservan `Mo Sold` y `Yr Sold`: se interpretan como la fecha de la valoración. Es un supuesto
  que debe revisarse en la evaluación ética final.

In [ ]:
pd.DataFrame({
    "Variables excluidas": COLUMNAS_EXCLUIDAS,
    "Presentes en el original": [c in load_raw().columns for c in COLUMNAS_EXCLUIDAS],
    "Presentes en X_train": [c in X_train.columns for c in COLUMNAS_EXCLUIDAS],
})

## 3. Métricas elegidas y por qué

| Métrica | Por qué se usa |
|---|---|
| **Cobertura ±10 %** | Es el KPI de negocio 1: porcentaje de viviendas con error absoluto relativo menor o igual a 10 %. |
| **Mejora del MAE vs. referencia** | Es el KPI de negocio 2: reducción del MAE frente a predecir la mediana del entrenamiento para todas las viviendas. |
| **MAE (USD)** | Magnitud promedio del error en las unidades originales: interpretable para quien valora una propiedad y base del KPI 2. |
| **RMSE (USD)** | Penaliza más los errores grandes; importante en viviendas caras, donde un error absoluto grande es más probable. |
| **R²** | Proporción de la variabilidad explicada respecto de predecir la media; permite comparar modelos en una escala común. |

Las metas son las del README (sección 4.1) y se declaran antes de entrenar:

In [ ]:
pd.DataFrame({
    "KPI": ["Cobertura de estimaciones dentro de la tolerancia", "Mejora del MAE respecto de la referencia (mediana)"],
    "Definición": [f"Error absoluto relativo <= {TOLERANCIA:.0%}", "1 - MAE modelo / MAE referencia"],
    "Meta": [f">= {META_COBERTURA:.0%}", f">= {META_MEJORA_MAE:.0%}"],
})

## 4. Referencia y modelo baseline

- **Referencia del KPI:** predecir la mediana de `SalePrice` del entrenamiento para todas las viviendas.
- **Baseline supervisado:** **Ridge** (regresión lineal con regularización L2). Es simple e interpretable, y la
  regularización controla la multicolinealidad entre predictores (por ejemplo `Garage Cars` y `Garage Area`).
- **Objetivo en escala logarítmica:** `SalePrice` es asimétrico a la derecha (ver la asimetría calculada abajo).
  El baseline aprende `log1p(SalePrice)` y devuelve predicciones en USD, de modo que todas las métricas
  siguen expresadas en dólares.
- **Validación:** 5 particiones con semilla fija; el preprocesamiento se ajusta dentro de cada partición.

El baseline se evalúa en validación cruzada y en entrenamiento. El conjunto de prueba se reserva
para la evaluación final de todos los modelos (sección 7).

In [ ]:
from src.train import (
    crear_cv, construir_pipeline, construir_ridge, referencia_mediana,
    validacion_cruzada, evaluar, evaluar_metas,
)

cv = crear_cv()
print(f"Asimetría de SalePrice en entrenamiento: {y_train.skew():.2f}")

# Referencia: mediana del entrenamiento
ref = referencia_mediana(X_train, y_train)
metricas_ref_cv, _ = validacion_cruzada(ref, X_train, y_train, cv)
mae_ref_cv = metricas_ref_cv["MAE"]

# Baseline: Ridge sobre log1p(SalePrice)
ridge_base = construir_pipeline(construir_ridge(alpha=1.0), X_train, log_objetivo=True)
metricas_ridge_cv, _ = validacion_cruzada(ridge_base, X_train, y_train, cv)
ridge_base.fit(X_train, y_train)
metricas_ridge_train = evaluar(ridge_base, X_train, y_train)

resultados_c10 = pd.DataFrame({
    "Referencia (mediana) - CV": metricas_ref_cv,
    "Ridge base - CV": metricas_ridge_cv,
    "Ridge base - entrenamiento": metricas_ridge_train,
}).T
display(resultados_c10.round(3))

metas_ridge_cv = evaluar_metas(metricas_ridge_cv, mae_ref_cv)
print(f"Mejora del MAE del baseline frente a la referencia (CV): {metas_ridge_cv['Mejora_MAE']:.1%} (meta >= {META_MEJORA_MAE:.0%})")
print(f"Cobertura ±10 % del baseline (CV): {metricas_ridge_cv['Cobertura_10']:.1%} (meta >= {META_COBERTURA:.0%}; se comprueba en prueba)")

## 5. Segundo modelo: gradient boosting

**Algoritmo:** `HistGradientBoostingRegressor`, un ensamble de árboles construidos de forma secuencial.
Es de otra familia que Ridge. Se eligió porque:

- Captura relaciones **no lineales** e **interacciones** (calidad, superficie, barrio y antigüedad actúan en conjunto sobre el precio) sin suponerlas.
- No exige una relación lineal con el objetivo ni sensibilidad a la escala de los predictores.
- Se entrena rápido, lo que permite buscar hiperparámetros con validación cruzada.
- Su comportamiento se interpreta mediante importancia por permutación (sección 7).

En una exploración previa, hecha solo con datos de entrenamiento, el boosting obtuvo menor MAE que Random Forest
con configuración por defecto; esa exploración no forma parte de este notebook.

Se entrena con el **mismo split, la misma validación cruzada y las mismas métricas** que el baseline.
Se usa el objetivo en USD, sin transformación logarítmica, porque los árboles no requieren esa corrección.

In [ ]:
from src.train import construir_boosting

boost_base = construir_pipeline(construir_boosting(), X_train)
metricas_boost_cv, _ = validacion_cruzada(boost_base, X_train, y_train, cv)
boost_base.fit(X_train, y_train)
metricas_boost_train = evaluar(boost_base, X_train, y_train)

resultados_c11 = pd.DataFrame({
    "Referencia (mediana) - CV": metricas_ref_cv,
    "Ridge base - CV": metricas_ridge_cv,
    "Boosting base - CV": metricas_boost_cv,
    "Ridge base - entrenamiento": metricas_ridge_train,
    "Boosting base - entrenamiento": metricas_boost_train,
}).T
display(resultados_c11.round(3))

## 6. Ajuste de hiperparámetros con validación cruzada

`GridSearchCV` sobre ambos modelos, con las mismas 5 particiones y el MAE como criterio
(`neg_mean_absolute_error`). **No se usa el conjunto de prueba.**

- **Ridge:** `alpha` en una escala logarítmica entre 0,1 y 100 (controla la fuerza de la regularización).
- **Boosting:** tasa de aprendizaje, profundidad máxima de los árboles, número de iteraciones y regularización L2.
  Combinar menor tasa de aprendizaje con más iteraciones y limitar la profundidad busca reducir el sobreajuste.

Limitación: no se usa validación cruzada anidada, por lo que el MAE de validación de los modelos ajustados
puede ser levemente optimista.

In [ ]:
from src.train import GRILLA_RIDGE, GRILLA_BOOSTING, buscar_hiperparametros

busqueda_ridge = buscar_hiperparametros(ridge_base, GRILLA_RIDGE, X_train, y_train, cv)
busqueda_boost = buscar_hiperparametros(boost_base, GRILLA_BOOSTING, X_train, y_train, cv)

display(pd.DataFrame({
    "Modelo": ["Ridge", "Boosting"],
    "Combinaciones evaluadas": [len(busqueda_ridge.cv_results_["params"]), len(busqueda_boost.cv_results_["params"])],
    "Mejores hiperparámetros": [busqueda_ridge.best_params_, busqueda_boost.best_params_],
    "MAE CV (USD)": [-busqueda_ridge.best_score_, -busqueda_boost.best_score_],
}).round(1))

# Métricas completas de los modelos ajustados (predicciones fuera de muestra)
metricas_ridge_aj_cv, _ = validacion_cruzada(busqueda_ridge.best_estimator_, X_train, y_train, cv)
metricas_boost_aj_cv, _ = validacion_cruzada(busqueda_boost.best_estimator_, X_train, y_train, cv)

display(pd.DataFrame({
    "Ridge base": metricas_ridge_cv, "Ridge ajustado": metricas_ridge_aj_cv,
    "Boosting base": metricas_boost_cv, "Boosting ajustado": metricas_boost_aj_cv,
}).T.round(3))

## 7. Comparación final, importancia de variables y análisis de errores

**Criterio de selección (definido antes de ver la prueba):** entre los modelos ajustados, gana el de
**menor MAE en validación cruzada**. Recién entonces se evalúan todos los modelos en el conjunto de prueba,
una sola vez, para informar el desempeño y comprobar las metas.

In [ ]:
from src.train import fila_resultado, importancia_permutacion, coeficientes_ridge

# Referencia en prueba (mediana del entrenamiento)
metricas_ref_test = evaluar(ref, X_test, y_test)
mae_ref_test = metricas_ref_test["MAE"]

modelos = {
    "Ridge base": dict(est=ridge_base, familia="Lineal regularizado (log1p del objetivo)",
                       hiper={"alpha": 1.0}, cv=metricas_ridge_cv),
    "Ridge ajustado": dict(est=busqueda_ridge.best_estimator_, familia="Lineal regularizado (log1p del objetivo)",
                           hiper=busqueda_ridge.best_params_, cv=metricas_ridge_aj_cv),
    "Boosting base": dict(est=boost_base, familia="Ensamble: gradient boosting",
                          hiper={"configuracion": "por defecto"}, cv=metricas_boost_cv),
    "Boosting ajustado": dict(est=busqueda_boost.best_estimator_, familia="Ensamble: gradient boosting",
                              hiper=busqueda_boost.best_params_, cv=metricas_boost_aj_cv),
}

candidatos = ["Ridge ajustado", "Boosting ajustado"]
nombre_ganador = min(candidatos, key=lambda n: modelos[n]["cv"]["MAE"])
modelo_ganador = modelos[nombre_ganador]["est"]

filas = [fila_resultado("Referencia (mediana)", "Predicción constante", {"valor": "mediana de SalePrice del entrenamiento"},
                        metricas_ref_cv, metricas_ref_test)]
for nombre, m in modelos.items():
    m["prueba"] = evaluar(m["est"], X_test, y_test)
    filas.append(fila_resultado(nombre, m["familia"], m["hiper"], m["cv"], m["prueba"],
                                mae_ref_test, seleccionado=(nombre == nombre_ganador)))
tabla_resultados = pd.DataFrame(filas)

print(f"Modelo ganador: {nombre_ganador} (menor MAE en validación cruzada).")
display(tabla_resultados.drop(columns=["familia", "hiperparametros"]).round(3))

In [ ]:
# Gráfico de comparación de modelos
orden = list(tabla_resultados["modelo"])
x = np.arange(len(orden))
ancho = 0.38

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.8))

ejes[0].bar(x - ancho / 2, tabla_resultados["MAE_CV"], ancho, label="Validación cruzada", color="#1E2761")
ejes[0].bar(x + ancho / 2, tabla_resultados["MAE_prueba"], ancho, label="Prueba", color="#B85042")
ejes[0].set_title("MAE por modelo (USD)")
ejes[0].set_ylabel("MAE (USD)")

ejes[1].bar(x - ancho / 2, tabla_resultados["Cobertura10_CV"] * 100, ancho, label="Validación cruzada", color="#1E2761")
ejes[1].bar(x + ancho / 2, tabla_resultados["Cobertura10_prueba"] * 100, ancho, label="Prueba", color="#B85042")
ejes[1].axhline(META_COBERTURA * 100, color="gray", linestyle="--", label=f"Meta ({META_COBERTURA:.0%})")
ejes[1].set_title("Predicciones con error relativo <= 10 % (%)")
ejes[1].set_ylabel("Cobertura (%)")

for eje in ejes:
    eje.set_xticks(x)
    eje.set_xticklabels(orden, rotation=25, ha="right")
    eje.legend()

fig.tight_layout()
fig.savefig(IMAGES_DIR / "modelos_comparacion.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Importancia de variables: permutación sobre las variables originales (modelo ganador, prueba)
# y coeficientes del Ridge ajustado como referencia interpretable.
imp = importancia_permutacion(modelo_ganador, X_test, y_test, n_repeats=10)
coef = coeficientes_ridge(modelos["Ridge ajustado"]["est"])

fig, ejes = plt.subplots(1, 2, figsize=(14, 6))

top_imp = imp.head(15).iloc[::-1]
ejes[0].barh(top_imp["Variable"], top_imp["Aumento_MAE"], xerr=top_imp["Desv"], color="#1E2761")
ejes[0].set_title(f"Importancia por permutación - {nombre_ganador}")
ejes[0].set_xlabel("Aumento del MAE al permutar la variable (USD)")

top_coef = coef.head(15).iloc[::-1]
colores = ["#B85042" if c < 0 else "#1E2761" for c in top_coef["Coeficiente"]]
ejes[1].barh(top_coef["Variable"], top_coef["Coeficiente"], color=colores)
ejes[1].set_title("Mayores coeficientes - Ridge ajustado")
ejes[1].set_xlabel("Coeficiente (variables estandarizadas, escala log de SalePrice)")

fig.tight_layout()
fig.savefig(IMAGES_DIR / "modelos_importancia.png", dpi=150, bbox_inches="tight")
plt.show()

display(imp.head(10).round(1))

In [ ]:
# Análisis de errores del modelo ganador en prueba
pred_test = modelo_ganador.predict(X_test)
resultados_prueba = pd.DataFrame({
    "y_real": y_test.to_numpy(), "y_pred": pred_test,
}, index=X_test.index)
resultados_prueba["residuo"] = resultados_prueba["y_pred"] - resultados_prueba["y_real"]
resultados_prueba["error_rel"] = resultados_prueba["residuo"].abs() / resultados_prueba["y_real"]

fig, ejes = plt.subplots(1, 3, figsize=(16, 4.6))

ejes[0].scatter(resultados_prueba["y_real"], resultados_prueba["y_pred"], s=14, alpha=0.6, color="#1E2761")
limite = [resultados_prueba[["y_real", "y_pred"]].min().min(), resultados_prueba[["y_real", "y_pred"]].max().max()]
ejes[0].plot(limite, limite, color="#B85042", linestyle="--")
ejes[0].set_title("Precio real vs. predicho (prueba)")
ejes[0].set_xlabel("Precio real (USD)")
ejes[0].set_ylabel("Precio predicho (USD)")

ejes[1].scatter(resultados_prueba["y_pred"], resultados_prueba["residuo"], s=14, alpha=0.6, color="#1E2761")
ejes[1].axhline(0, color="#B85042", linestyle="--")
ejes[1].set_title("Residuos vs. precio predicho")
ejes[1].set_xlabel("Precio predicho (USD)")
ejes[1].set_ylabel("Residuo (predicho - real, USD)")

ejes[2].hist(resultados_prueba["error_rel"].clip(upper=0.5) * 100, bins=40, color="#1E2761")
ejes[2].axvline(10, color="#B85042", linestyle="--", label="Tolerancia 10 %")
ejes[2].set_title("Error relativo absoluto (%, truncado en 50)")
ejes[2].set_xlabel("Error relativo (%)")
ejes[2].legend()

fig.tight_layout()
fig.savefig(IMAGES_DIR / "modelos_residuos.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Error por rango de precio y por barrio (se informa el tamaño de cada grupo)
from src.train import rangos_precio, error_por_grupo

resultados_prueba["rango_precio"] = rangos_precio(y_train, resultados_prueba["y_real"])
resultados_prueba["Neighborhood"] = X_test["Neighborhood"]

por_rango = error_por_grupo(resultados_prueba, "rango_precio")
por_barrio = error_por_grupo(resultados_prueba, "Neighborhood", min_n=10)

print("Error por rango de precio (cuartiles del entrenamiento):")
display(por_rango.round(3))
print("Error por barrio (la columna Muestra_suficiente marca grupos con 10 o más viviendas de prueba):")
display(por_barrio.round(3))

barrios_ok = por_barrio[por_barrio["Muestra_suficiente"]].sort_values("Cobertura_10")
fig, eje = plt.subplots(figsize=(9, 6))
eje.barh(barrios_ok["Neighborhood"], barrios_ok["Cobertura_10"] * 100, color="#1E2761")
eje.axvline(META_COBERTURA * 100, color="#B85042", linestyle="--", label=f"Meta global ({META_COBERTURA:.0%})")
eje.set_title("Cobertura ±10 % por barrio (grupos con 10 o más viviendas de prueba)")
eje.set_xlabel("Cobertura (%)")
eje.legend()
fig.tight_layout()
fig.savefig(IMAGES_DIR / "modelos_error_barrio.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Mayores errores: Sale Condition se usa solo como diagnóstico, no como predictor
crudo = load_raw(keep_default_na=False, na_values=[""])
diagnostico = resultados_prueba.assign(
    Sale_Condition=crudo.loc[X_test.index, "Sale Condition"],
    Gr_Liv_Area=X_test["Gr Liv Area"],
)
orden_error = diagnostico["residuo"].abs().sort_values(ascending=False).index
display(diagnostico.loc[orden_error, ["y_real", "y_pred", "residuo", "Neighborhood", "Sale_Condition", "Gr_Liv_Area"]].head(8).round(0))

In [ ]:
fila_ganador = tabla_resultados[tabla_resultados["modelo"] == nombre_ganador].iloc[0]
print(f"Modelo ganador: {nombre_ganador}")
print(f"  MAE en validación cruzada: {fila_ganador['MAE_CV']:,.0f} USD | en prueba: {fila_ganador['MAE_prueba']:,.0f} USD")
print(f"  Mejora del MAE vs. referencia (prueba): {fila_ganador['Mejora_MAE_prueba']:.1%} -> meta >= {META_MEJORA_MAE:.0%}: "
      f"{'CUMPLE' if fila_ganador['Cumple_meta_mejora'] else 'NO CUMPLE'}")
print(f"  Cobertura ±10 % (prueba): {fila_ganador['Cobertura10_prueba']:.1%} -> meta >= {META_COBERTURA:.0%}: "
      f"{'CUMPLE' if fila_ganador['Cumple_meta_cobertura'] else 'NO CUMPLE'}")
print("Las metas no se modifican a partir de estos resultados.")

### Lectura inicial de los resultados

- **Selección del modelo:** el ganador se eligió por su MAE en validación cruzada, antes de mirar la prueba. La prueba solo informa el desempeño.
- **Validación cruzada frente a prueba:** una diferencia grande entre ambas señalaría sobreajuste o una prueba poco representativa; el EDA de la Parte 1 usó el dataset completo, de modo que la prueba no estuvo totalmente aislada de la exploración.
- **Errores por grupo:** un promedio global aceptable puede ocultar barrios o rangos de precio con peor desempeño. Los grupos con pocas viviendas de prueba no permiten conclusiones firmes y se informan con su tamaño.
- **Mayores errores:** se revisan junto con `Sale Condition` y la superficie para saber si corresponden a ventas atípicas (por ejemplo, ventas parciales) y no a fallos generales del modelo.
- **Importancia de variables:** los resultados describen asociaciones predictivas del modelo, no efectos causales de las características sobre el precio.
- Las recomendaciones de negocio corresponden a la Parte 3 y no se formulan aquí.